# Purchase Intent Prediction Hackathon

## The business problem

An e-commerce company can offer live assistance or a targeted incentive to only a limited number of active visitors. Your task is to build a classical machine-learning model that assigns every session a probability of ending in a purchase.

Your model should answer:

$$
P(\text{purchase}=1\mid\text{session information})
$$

### Success criteria

- **Leaderboard metric:** Average Precision (AP), higher is better.
- **Minimum qualifying AP:** `0.55` on the hidden test set.
- **Tie-breaker:** ROC-AUC.
- **Business task:** choose a threshold that maximizes recall while maintaining precision of at least 40% on your validation data.

Read `RULES.md`, `METRIC_GUIDE.md`, and `PRESENTATION_GUIDE.md` before starting.


## Suggested two-hour workflow

| Time | Milestone |
|---|---|
| 0–20 min | Inspect the target, features and class balance |
| 20–40 min | Create a validation strategy and baseline |
| 40–75 min | Compare two or three suitable models |
| 75–100 min | Improve the best approach with focused experiments |
| 100–110 min | Select and explain a business threshold |
| 110–120 min | Train the final model and create the submission |

**Team principle:** make one controlled change per experiment and record the result.


## 1. Setup


In [ ]:
# Google Colab normally includes these packages.
# Run this only if an import fails.
# %pip install -q pandas numpy scikit-learn matplotlib seaborn xgboost


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split


RANDOM_STATE = 42
TARGET = "Revenue"
ID_COLUMN = "session_id"
MINIMUM_AP = 0.55


## 2. Load the competition data

Keep `train.csv` and `test.csv` in the same directory as this notebook. The training file contains the target. The test target is hidden from you.


In [ ]:
DATA_DIR = Path(".")
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print("Train shape:", train.shape)
print("Test shape: ", test.shape)
display(train.head())


In [ ]:
assert TARGET in train.columns
assert TARGET not in test.columns
assert ID_COLUMN in train.columns and ID_COLUMN in test.columns
assert train[ID_COLUMN].is_unique and test[ID_COLUMN].is_unique
assert set(train[ID_COLUMN]).isdisjoint(set(test[ID_COLUMN]))

print("Data files passed the basic checks.")


## 3. Understand the features

| Group | Columns | Meaning |
|---|---|---|
| Page counts | `Administrative`, `Informational`, `ProductRelated` | Number of pages of each kind visited |
| Page durations | matching `_Duration` columns | Time spent on those page groups |
| Behaviour rates | `BounceRates`, `ExitRates` | Session-level navigation behaviour |
| Page value | `PageValues` | Estimated value of pages viewed before transaction completion |
| Calendar | `Month`, `Weekend`, `SpecialDay` | Timing and proximity to special days |
| Technical/context | `OperatingSystems`, `Browser`, `Region`, `TrafficType` | Encoded visitor context |
| Visitor status | `VisitorType` | New, returning or other visitor |
| Target | `Revenue` | Whether the session ended in a purchase |

Some integer-looking columns are **category codes**, not quantities. Treating browser `8` as twice browser `4` would introduce an artificial ordering.

**TODO:** inspect data types, unique values, distributions, unusual values and relationships with the target. Keep your exploration focused on decisions that may improve the model.


In [ ]:
# TODO: inspect columns, data types, missing values and unique values.
# Useful starting points: train.info(), train.describe(), value_counts().


In [ ]:
purchase_rate = train[TARGET].mean()
print(f"Purchase rate: {purchase_rate:.2%}")

ax = train[TARGET].value_counts().sort_index().plot(
    kind="bar", color=["#4C78A8", "#E45756"], figsize=(6, 4)
)
ax.set_xticklabels(["No purchase", "Purchase"], rotation=0)
ax.set_ylabel("Sessions")
ax.set_title("Target distribution")
plt.show()


## 4. Create one trustworthy validation split

Use stratification so that training and validation contain similar purchase rates. Do not repeatedly change the split to obtain a more favourable score.

The ID identifies a row; it must not be used as a predictive feature.


In [ ]:
# TODO: create X and y, then make the validation split.
# Minimal hints:
# - Remove both Revenue and session_id from the model features.
# - Keep 20% of the labelled rows for validation.
# - Preserve the target proportion using stratification.
# - Use RANDOM_STATE so the experiment is reproducible.
# - Name the outputs X_train, X_valid, y_train and y_valid;
#   later sections use these names.


## 5. Evaluation helpers

Submit **probabilities**, not class predictions. Average Precision rewards a model that places actual purchasers near the top of its ranking.

A random ranking has expected AP close to the positive-class prevalence, not 0.50.


In [ ]:
def evaluate_probabilities(y_true, probabilities, label="model"):
    probabilities = np.asarray(probabilities)
    metrics = {
        "model": label,
        "average_precision": average_precision_score(y_true, probabilities),
        "roc_auc": roc_auc_score(y_true, probabilities),
    }
    print(
        f"{label}: AP={metrics['average_precision']:.4f} | "
        f"ROC-AUC={metrics['roc_auc']:.4f}"
    )
    return metrics


experiment_results = []


## 6. Establish a no-skill baseline

This baseline predicts the same purchase prevalence for every session. Your useful models should beat it comfortably.


In [ ]:
dummy = DummyClassifier(strategy="prior")
dummy.fit(X_train, y_train)
dummy_probabilities = dummy.predict_proba(X_valid)[:, 1]

experiment_results.append(
    evaluate_probabilities(y_valid, dummy_probabilities, "Dummy baseline")
)


## 7. Build a leakage-safe preprocessing pipeline

Your pipeline should learn preprocessing only from `X_train`.

Consider:

- Which columns are genuinely numerical?
- Which integer columns represent categories?
- Which models need scaling?
- How will unseen categories be handled?

**TODO:** create suitable feature groups and a `ColumnTransformer`. Use a `Pipeline` so validation data is never used to fit preprocessing.


In [ ]:
# TODO: define numerical and categorical columns.
# TODO: build a ColumnTransformer.
# TODO: combine preprocessing and a first model in a Pipeline.


## 8. Controlled model experiments

Compare a small number of justified approaches. Suitable families include logistic regression, decision trees, random forests, gradient boosting and XGBoost.

For every experiment, record:

- what changed;
- why you expected it to help;
- validation AP and ROC-AUC;
- what you learned.

Do not tune twenty parameters at once. Spend most of your time improving the strongest approach.


In [ ]:
# TODO: run your first controlled model experiment.
# Minimal hint: train on the training partition, obtain positive-class
# probabilities for validation, and record them with the evaluation helper.


In [ ]:
# TODO: train one or two additional, purposeful experiments.


In [ ]:
pd.DataFrame(experiment_results).sort_values(
    "average_precision", ascending=False
)


## 9. Feature engineering and imbalance experiments

Try only ideas that make sense for browsing behaviour. Examples to investigate rather than blindly add:

- total pages or total duration;
- duration per page with safe division;
- whether any page value is present;
- interactions involving returning visitors or special days;
- class weights or sampling methods applied only inside training folds.

**Important:** resampling must never be performed before the train/validation split.


In [ ]:
# TODO: add one or two justified feature or imbalance experiments.
# Record whether each change actually improves validation AP.


## 10. Choose a business threshold

AP ranks probabilities and does not require a threshold. A business action does.

Find a threshold that maximizes recall while keeping validation precision at or above 40%:

$$
\max_t\;\operatorname{Recall}(t)
\quad\text{subject to}\quad
\operatorname{Precision}(t)\ge 0.40
$$

Explain the consequence: among all contacted sessions, at least 40% should purchase, while reaching as many purchasers as possible.


In [ ]:
def plot_precision_recall_tradeoff(y_true, probabilities):
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)

    plt.figure(figsize=(7, 4))
    plt.plot(thresholds, precision[:-1], label="Precision")
    plt.plot(thresholds, recall[:-1], label="Recall")
    plt.axhline(0.40, color="black", linestyle="--", label="40% precision")
    plt.xlabel("Decision threshold")
    plt.ylabel("Metric value")
    plt.ylim(0, 1.02)
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()


# TODO: call this with the validation probabilities from your best model.
# TODO: select and justify a threshold satisfying the business constraint.


## 11. Inspect errors and important features

Your presentation must include at least one explanation of model behaviour.

Depending on the model, consider coefficients, impurity-based importance, permutation importance, SHAP, or focused error analysis. State the limitations of your chosen interpretation method.


In [ ]:
# TODO: inspect the best model and a few false positives/false negatives.


## 12. Train the final model

Once the approach is frozen:

1. rebuild it with the chosen settings;
2. train it on all rows in `train.csv`;
3. predict purchase probabilities for `test.csv`;
4. do not tune further using the hidden leaderboard.


In [ ]:
X_full = train.drop(columns=[TARGET, ID_COLUMN])
y_full = train[TARGET].astype(int)
X_test = test.drop(columns=[ID_COLUMN])

# TODO: assign your final pipeline/model to final_model.
# final_model.fit(X_full, y_full)
# test_probabilities = final_model.predict_proba(X_test)[:, 1]


## 13. Create and validate the submission

The required columns are exactly:

```text
session_id,purchase_probability
```

Probabilities must be finite and between 0 and 1. Do not submit hard class labels.


In [ ]:
def validate_submission(submission, expected_ids):
    required = [ID_COLUMN, "purchase_probability"]
    assert list(submission.columns) == required, f"Expected columns: {required}"
    assert submission[ID_COLUMN].is_unique, "Duplicate session IDs found"
    assert set(submission[ID_COLUMN]) == set(expected_ids), "Test IDs do not match"
    assert submission["purchase_probability"].notna().all(), "Missing probabilities"
    assert np.isfinite(submission["purchase_probability"]).all(), "Non-finite values"
    assert submission["purchase_probability"].between(0, 1).all(), "Outside [0, 1]"
    if submission["purchase_probability"].nunique() <= 2:
        print("Warning: very few unique probabilities; confirm these are not hard labels.")
    print("Submission passed all checks.")


# Uncomment after creating test_probabilities.
# submission = pd.DataFrame({
#     ID_COLUMN: test[ID_COLUMN],
#     "purchase_probability": test_probabilities,
# })
# validate_submission(submission, test[ID_COLUMN])
# submission.to_csv("TEAM_NAME_submission.csv", index=False)
# display(submission.head())


## 14. Final team notes

Complete these before presenting:

- **Best validation AP:**
- **Hidden test AP:**
- **Best model and preprocessing:**
- **Most useful experiment:**
- **Experiment that failed:**
- **Chosen threshold:**
- **Validation precision and recall at that threshold:**
- **Most important model insight:**
- **What we would try with one more hour:**
